In [4]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies_data = pd.read_csv(r"C:\Users\Anjali\Desktop\AI_Course\IIT Delhi\Project\Capstone Project Problem Statements, Submission Guidelines & Evaluation Procedure\AI-ML-DL_Projects\Movie_Recommender\data\movies.csv")
ratings_data = pd.read_csv(r"C:\Users\Anjali\Desktop\AI_Course\IIT Delhi\Project\Capstone Project Problem Statements, Submission Guidelines & Evaluation Procedure\AI-ML-DL_Projects\Movie_Recommender\data\ratings.csv")

df = pd.merge(ratings_data, movies_data, on='movieId')

df.head()



,userId,movieId,rating,timestamp,title,genres
0,1,296,5.0,1147880044,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
1,1,306,3.5,1147868817,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama
2,1,307,5.0,1147868828,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama
3,1,665,5.0,1147878820,Underground (1995),Comedy|Drama|War
4,1,899,3.5,1147868510,Singin' in the Rain (1952),Comedy|Musical|Romance


In [5]:
%pip install scikit-surprise

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 8.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [11]:
from xarray import Dataset
from surprise.model_selection import train_test_split, cross_validate

# 4. The Singular Value Decomposition algorithm and metrics
from surprise import SVD
from surprise import accuracy
from surprise import Reader, Dataset

movie_summary = df.groupby('movieId').agg(
    Avg_Rating=('rating', 'mean'),
    Total_Votes=('rating', 'count')
).reset_index()

print("--- Pandas Global Average Summary ---")
print(movie_summary.round(2).to_string(index=False))
print("\n" + "="*40 + "\n")

# 3. Prepare data for the Surprise Recommender Library
# Define the rating scale limits (min, max)
reader = Reader(rating_scale=(1.0, 5.0))
data = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)

# Split data into training and testing sets (80% train, 20% test)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# 4. Initialize and train SVD (Singular Value Decomposition)
# SVD automatically builds mathematical baselines using global, movie, and user rating averages
algo = SVD()
algo.fit(trainset)

# 5. Evaluate the model predictions against the test set
predictions = algo.test(testset)
rmse_score = accuracy.rmse(predictions, verbose=False)

print(f"Model Training Complete. Test Set RMSE Error: {rmse_score:.4f}")

# 6. Predict what rating User 2 would give to 'Interstellar' (which they haven't watched)
target_user = 2
target_movie = 'Interstellar'
pred = algo.predict(target_user, target_movie)

print(f"\n--- AI Predicted Rating ---")
print(f"User {target_user} is predicted to rate '{target_movie}' as: {pred.est:.2f} out of 5")

--- Pandas Global Average Summary ---
 movieId  Avg_Rating  Total_Votes
       1        3.89        57309
       2        3.25        24228
       3        3.14        11804
       4        2.85         2523
       5        3.06        11714
       6        3.85        24588
       7        3.36        12132
       8        3.11         1344
       9        2.99         3711
      10        3.42        28265
      11        3.66        17042
      12        2.62         3741
      13        3.33         1715
      14        3.42         5509
      15        2.72         2760
      16        3.82        18404
      17        3.95        19729
      18        3.38         5374
      19        2.64        21552
      20        2.87         3840
      21        3.57        22277
      22        3.32         9237
      23        3.15         4108
      24        3.19         7450
      25        3.68        20070
      26        3.61         2549
      27        3.40         1577
      28  